In [1]:
import pandas as pd

wines = pd.read_csv('winemag-data-130k-v2.csv', index_col=0)

wines.head()

,country,description,designation,points,price,province,region_1,region_2,taster_name,taster_twitter_handle,title,variety,winery
0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Etna,NaN,Kerin O’Keefe,@kerinokeefe,Nicosia 2013 Vulkà Bianco (Etna),White Blend,Nicosia
1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,NaN,NaN,Roger Voss,@vossroger,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,Quinta dos Avidagos
2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,Rainstorm
3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,Lake Michigan Shore,NaN,Alexander Peartree,NaN,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,St. Julian
4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Willamette Valley,Willamette Valley,Paul Gregutt,@paulgwine,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,Sweet Cheeks


## Remove duplicate rows

In [2]:
print(f'Dataframe size: {wines.shape}')
print(f'{wines.duplicated().sum()} duplicate rows')

wines = wines.drop_duplicates()
print(f'New dataframe size: {wines.shape}')

Dataframe size: (129971, 13)
9983 duplicate rows
New dataframe size: (119988, 13)


## Drop columns and clean description data

In [3]:
import nltk
import string
from nltk.corpus import stopwords

nltk.download('stopwords')
stop_words = set(stopwords.words('english'))


stemmer = nltk.PorterStemmer()
translator = str.maketrans('', '', string.punctuation)

cleaned = wines.drop(columns=['region_1', 'region_2', 'taster_twitter_handle', 'taster_name', 'winery'])

def clean_descriptions(description: str):
    puncs_removed = description.translate(translator)
    tokens = puncs_removed.lower().split()
    stemmed_tokens = list(map(stemmer.stem, tokens))
    stop_words_removed = ' '.join([token for token in stemmed_tokens if token not in stop_words])
    return stop_words_removed
 


cleaned['taste_profile'] = cleaned['description'].apply(clean_descriptions)
cleaned.head()


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/kallevapaa/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


,country,description,designation,points,price,province,title,variety,taste_profile
0,Italy,"Aromas include tropical fruit, broom, brimston...",Vulkà Bianco,87,NaN,Sicily & Sardinia,Nicosia 2013 Vulkà Bianco (Etna),White Blend,aroma includ tropic fruit broom brimston dri h...
1,Portugal,"This is ripe and fruity, a wine that is smooth...",Avidagos,87,15.0,Douro,Quinta dos Avidagos 2011 Avidagos Red (Douro),Portuguese Red,thi ripe fruiti wine smooth still structur fir...
2,US,"Tart and snappy, the flavors of lime flesh and...",NaN,87,14.0,Oregon,Rainstorm 2013 Pinot Gris (Willamette Valley),Pinot Gris,tart snappi flavor lime flesh rind domin green...
3,US,"Pineapple rind, lemon pith and orange blossom ...",Reserve Late Harvest,87,13.0,Michigan,St. Julian 2013 Reserve Late Harvest Riesling ...,Riesling,pineappl rind lemon pith orang blossom start a...
4,US,"Much like the regular bottling from 2012, this...",Vintner's Reserve Wild Child Block,87,65.0,Oregon,Sweet Cheeks 2012 Vintner's Reserve Wild Child...,Pinot Noir,much like regular bottl 2012 thi come across r...


## Combine rows containing data for same wine

In [4]:
agg_funcs = {
    'country': 'first',
    'description': lambda x: ' '.join(x),
    'designation': 'first',
    'points': 'mean',
    'price': 'mean',
    'province': 'first',
    'variety': 'first',
    'taste_profile': lambda x: ' '.join(x)
}

aggregated = cleaned.groupby('title', as_index=False).agg(agg_funcs)
aggregated.head()

,title,country,description,designation,points,price,province,variety,taste_profile
0,1+1=3 2008 Rosé Cabernet Sauvignon (Penedès),Spain,The previous two years we did not find this wi...,Rosé,82.0,18.0,Catalonia,Cabernet Sauvignon,previou two year find thi wine worthi rate thi...
1,1+1=3 NV Brut Sparkling (Cava),Spain,Spiced apple and toast aromas are clean and di...,Brut,87.0,16.0,Catalonia,Sparkling Blend,spice appl toast aroma clean direct palat mali...
2,1+1=3 NV Cygnus Brut Nature Reserva Made With ...,Spain,"Clean, fresh apple aromas and a minerally, cit...",Cygnus Brut Nature Reserva Made With Organic G...,89.0,20.0,Catalonia,Sparkling Blend,clean fresh appl aroma miner citric palat poin...
3,1+1=3 NV Rosé Sparkling (Cava),Spain,"A dusty, yeasty nose is simplistic but friendl...",Rosé,86.0,20.0,Catalonia,Sparkling Blend,dusti yeasti nose simplist friendli palat thi ...
4,10 Knots 2006 Beachcomber White (Paso Robles),US,"A Rhône blend of Viognier, Roussanne and Marsa...",Beachcomber,83.0,21.0,California,Rhône-style White Blend,rhône blend viognier roussann marsann tast swe...


View aggregation result for the most frequently occurring wine title:

In [5]:
title_mode = cleaned['title'].mode()[0]
cleaned[cleaned['title'] == title_mode]

,country,description,designation,points,price,province,title,variety,taste_profile
3209,US,"Creamy, lush and somewhat robust, this dry spa...",Sonoma Brut,90,22.0,California,Gloria Ferrer NV Sonoma Brut Sparkling (Sonoma...,Sparkling Blend,creami lush somewhat robust thi dri sparkler o...
4399,US,"Made predominantly from Pinot Noir, this is an...",Sonoma Brut,88,22.0,California,Gloria Ferrer NV Sonoma Brut Sparkling (Sonoma...,Sparkling Blend,made predominantli pinot noir thi easygo appro...
27773,US,A wonderfully drinkable sparkling wine that ap...,Sonoma Brut,90,20.0,California,Gloria Ferrer NV Sonoma Brut Sparkling (Sonoma...,Sparkling Blend,wonder drinkabl sparkl wine appeal immedi bala...
63179,US,"Made from mostly Pinot Noir grapes, with an ad...",Sonoma Brut,92,22.0,California,Gloria Ferrer NV Sonoma Brut Sparkling (Sonoma...,Sparkling Blend,made mostli pinot noir grape addit 88 chardonn...
81563,US,This wine shows lots of finesse for the price....,Sonoma Brut,89,24.0,California,Gloria Ferrer NV Sonoma Brut Sparkling (Sonoma...,Sparkling Blend,thi wine show lot finess price mouss except re...
94321,US,"A bit rough and scouring in texture, this tast...",Sonoma Brut,85,20.0,California,Gloria Ferrer NV Sonoma Brut Sparkling (Sonoma...,Sparkling Blend,bit rough scour textur thi tast sweet orang st...
100738,US,"A good, dry and elegant bubbly. Shows crisp fl...",Sonoma Brut,89,20.0,California,Gloria Ferrer NV Sonoma Brut Sparkling (Sonoma...,Sparkling Blend,good dri eleg bubbl show crisp flavor yeasti b...
109001,US,From almost 92% Pinot Noir with the remainder ...,Sonoma Brut,88,22.0,California,Gloria Ferrer NV Sonoma Brut Sparkling (Sonoma...,Sparkling Blend,almost 92 pinot noir remaind chardonnay thi dr...
122208,US,This bubbly is rich in cherry and raspberry fr...,Sonoma Brut,86,20.0,California,Gloria Ferrer NV Sonoma Brut Sparkling (Sonoma...,Sparkling Blend,thi bubbl rich cherri raspberri fruit addit us...


In [6]:
aggregated[aggregated['title'] == title_mode]

,title,country,description,designation,points,price,province,variety,taste_profile
51652,Gloria Ferrer NV Sonoma Brut Sparkling (Sonoma...,US,"Creamy, lush and somewhat robust, this dry spa...",Sonoma Brut,88.555556,21.333333,California,Sparkling Blend,creami lush somewhat robust thi dri sparkler o...


## Wine recommender

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

vectorizer = TfidfVectorizer(
    max_df=0.5,
    min_df=5,
    ngram_range=(1,2)
)


# use a random sample from the wine dataframe to limit the number of wines
test_df = aggregated.sample(10000, random_state=23, ignore_index=True)

tfidf_matrix = vectorizer.fit_transform(aggregated['taste_profile'])
tfidf_matrix.shape

test_matrix = vectorizer.fit_transform(aggregated['taste_profile'])


(118840, 94958)

# Dump vectorizer and TF-IDF matrix to files so they can be used by our API-endpoint

In [8]:
import joblib
from scipy import sparse

joblib.dump(vectorizer, "../assets/tfidf_vectorizer.joblib", compress=3)
sparse.save_npz("../assets/tfidf_matrix.npz", tfidf_matrix)
aggregated.to_csv('../assets/wines_data.csv')


### Version 1: user input is a wine title

In [ ]:
cosine_sim = linear_kernel(test_matrix, test_matrix)
cosine_sim.shape

In [ ]:
index = pd.Series(test_df.index, index=test_df['title']).drop_duplicates()

In [ ]:
def recommend_wine_by_title(title, cosine_sim=cosine_sim):
    idx = index[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:6]
    position = [i[0] for i in sim_scores]
    return test_df.iloc[position]

test_title = test_df.iloc[0, 0]
print(f'Input wine title: {test_title}')

recommendations = recommend_wine_by_title(test_title)
recommendations

In [ ]:
print('Input wine and description:')
print(test_title)
print(test_df[test_df['title'] == test_title].iloc[0,2])

In [ ]:
print('Top recommendation and description:')
print(recommendations.iloc[0,0])
print(recommendations.iloc[0,2])

### Version 2: user input is words describing taste

In [ ]:
def recommend_wine_by_description(description, vectorizer=vectorizer, tfidf_matrix=test_matrix):
    translator = str.maketrans('', '', string.punctuation)
    description = description.translate(translator)
    tokens = description.lower().split()
    stemmed = list(map(stemmer.stem, tokens))
    description = ' '.join(stemmed)

    tfidf_input = vectorizer.transform([description])
    cosine_sim = linear_kernel(tfidf_matrix, tfidf_input)

    sim_scores = list(enumerate(cosine_sim))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:6]
    position = [i[0] for i in sim_scores]

    return test_df.iloc[position]

user_input = "white flowers, mineral, peach"
recommendations = recommend_wine_by_description(user_input)
recommendations

In [ ]:
print(f'Top 3 recommendations for "{user_input}":\n')
for i in range(3):
    print(recommendations.iloc[i,0])
    print(recommendations.iloc[i,2])
    print()